# Data Cleaning

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import os
import pickle

In [71]:
# Establish directories
data_dir = f'{Path.cwd().parent}/data/'

# Import dataset
dataset = data_dir+'customer_churn.csv'
df = pd.read_csv(dataset, index_col=0).reset_index()
df.head()

,Customer_ID,Gender,Region,Subscription_Plan,Payment_Method,Age,Days_Since_Last_Login,Customer_Service_Calls,Monthly_Spend,Churn
0,10001,Male,East,Basic,PayPal,27.0,27,1,61.08,0
1,10002,Unknown,West,Pro,PayPal,55.0,146,2,73.09,0
2,10003,Female,West,Pro,E-Wallet,NaN,147,2,116.30,1
3,10004,Female,south,Basic,Credit Card,NaN,34,1,0.58,1
4,10005,Male,North,Basic,Crypto,30.0,146,3,46.35,1


In [72]:
# Dataset descriptive information
df.describe()

,Customer_ID,Age,Days_Since_Last_Login,Customer_Service_Calls,Monthly_Spend,Churn
count,10300.000000,9267.000000,10300.000000,10300.000000,9786.000000,10300.000000
mean,14859.237864,33.521852,89.449903,1.787379,300.093737,0.762039
std,2959.197492,11.514686,52.088510,1.363470,4733.134612,0.425856
min,10001.000000,-10.000000,0.000000,-1.000000,-10.000000,0.000000
25%,12275.750000,26.000000,44.000000,1.000000,26.275000,1.000000
50%,14850.500000,33.000000,90.000000,2.000000,66.355000,1.000000
75%,17425.250000,40.000000,135.000000,3.000000,115.915000,1.000000
max,20000.000000,150.000000,179.000000,9.000000,99999.990000,1.000000


In [73]:
# Dataset Information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer_ID             10300 non-null  int64  
 1   Gender                  10300 non-null  str    
 2   Region                  10300 non-null  str    
 3   Subscription_Plan       10300 non-null  str    
 4   Payment_Method          9575 non-null   str    
 5   Age                     9267 non-null   float64
 6   Days_Since_Last_Login   10300 non-null  int64  
 7   Customer_Service_Calls  10300 non-null  int64  
 8   Monthly_Spend           9786 non-null   float64
 9   Churn                   10300 non-null  int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 804.8 KB


In [74]:
# Checking for duplicates
df.duplicated().sum()

np.int64(300)

In [75]:
# Standardizing columns prior to data cleaning
df.columns = (
    df.columns.str.strip()
    .str.lower()
)
df.columns

Index(['customer_id', 'gender', 'region', 'subscription_plan',
       'payment_method', 'age', 'days_since_last_login',
       'customer_service_calls', 'monthly_spend', 'churn'],
      dtype='str')

In [76]:
# Checking for data accuracy (categorical variables)

cols = [
    'gender','region'
    ,'subscription_plan','payment_method'
]
cats = df.dropna().copy()
for col in cols:
    print(cats[col].unique())

<StringArray>
['Male', 'Unknown', 'Non-Binary', 'Female']
Length: 4, dtype: str
<StringArray>
['East', 'West', 'North', 'South', 'north', 'east', 'south', 'west']
Length: 8, dtype: str
<StringArray>
['Basic', 'Pro', 'BASIC', 'PLUS', 'Plus', 'PRO']
Length: 6, dtype: str
<StringArray>
['PayPal', 'Crypto', 'E-Wallet', 'Credit Card']
Length: 4, dtype: str


>---
**_Null Values_**
* Payment Method
* Age
* Monthly Spend
>---
**_Data Type Conversion_**

_Int_
* Age
* Customer Service Calls
>---
**_Invalid Values and Outliers_**
* Age
    - Negative Values
    - Invalid outlier age
* Customer Service Calls
    - -1 Calls (Invalid)
* Monthly Spend
    - Negative spending
    - Extreme outlier
>---
**_Duplicated Values_**
- 301 total duplicated records
>---
**_Note:_**

    Null Values will be dropped during analysis and prior to modeling after variable is determined to be statistically significant.

    Data Type conversion will also be done during the analysis which requires no null values
---

## Handling Age Values

In [77]:
# Age Descriptive Information
print(f'Age Descriptive Information\n{df.age.describe()}')
print(df.age.count())
# Valid Age: 18
df.age = np.where(df.age<18,np.nan,df.age)
print('\nUpdate')
print(df.age.describe())
print(df.age.count())

Age Descriptive Information
count    9267.000000
mean       33.521852
std        11.514686
min       -10.000000
25%        26.000000
50%        33.000000
75%        40.000000
max       150.000000
Name: age, dtype: float64
9267

Update
count    8738.000000
mean       34.798009
std        10.508642
min        18.000000
25%        28.000000
50%        34.000000
75%        41.000000
max       150.000000
Name: age, dtype: float64
8738


In [78]:
print('High Age Values:\n',df.age.sort_values(ascending=False).unique()[:10])
# Max valid age: 70
df.age = np.where(df.age>70, np.nan, df.age)
print(f'\nUpdate\n{df.age.describe()}')
print(df.age.count())

High Age Values:
 [150.  70.  67.  66.  65.  64.  63.  62.  61.  60.]

Update
count    8718.000000
mean       34.533723
std         8.953512
min        18.000000
25%        28.000000
50%        34.000000
75%        41.000000
max        70.000000
Name: age, dtype: float64
8718


## Handling Customer Service Calls

In [79]:
print(df.customer_service_calls.describe())
print(df.customer_service_calls[~df.customer_service_calls.isna()].count())
df.customer_service_calls = np.where(df.customer_service_calls<0, np.nan, df.customer_service_calls)
print(f'\nUpdate\n{df.customer_service_calls.describe()}')
print(df.customer_service_calls[~df.customer_service_calls.isna()].count())

count    10300.000000
mean         1.787379
std          1.363470
min         -1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          9.000000
Name: customer_service_calls, dtype: float64
10300

Update
count    10248.000000
mean         1.801522
std          1.352352
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          9.000000
Name: customer_service_calls, dtype: float64
10248


## Handling Monthly Spend

In [80]:
df.monthly_spend.describe()

count     9786.000000
mean       300.093737
std       4733.134612
min        -10.000000
25%         26.275000
50%         66.355000
75%        115.915000
max      99999.990000
Name: monthly_spend, dtype: float64

In [81]:
# Converting negative values to nan
df.monthly_spend = np.where(df.monthly_spend<0, np.nan, df.monthly_spend)
df.monthly_spend.describe()

count     9737.000000
mean       301.654237
std       4744.979031
min          0.000000
25%         26.830000
50%         66.670000
75%        116.230000
max      99999.990000
Name: monthly_spend, dtype: float64

In [82]:
# Identifying valid spend
print('High Spendings\n',df.monthly_spend.sort_values(ascending=False).unique()[:10])
# Maximum monthly spend: 311.39
df.monthly_spend = np.where(df.monthly_spend>311.39, np.nan, df.monthly_spend)
print('\nUpdate\n',df.monthly_spend.describe())

High Spendings
 [99999.99   311.39   289.59   283.01   277.22   276.7    271.97   269.21
   264.9    263.82]

Update
 count    9715.000000
mean       75.883431
std        59.568490
min         0.000000
25%        26.735000
50%        66.520000
75%       115.825000
max       311.390000
Name: monthly_spend, dtype: float64


## Handling Data Quality Issues

In [89]:
for col in cols:
    df[col]=(
        df[col]
        .str.strip()
        .str.replace(' ','_')
        .str.lower()
    )
for col in cols:
    print(col)
    print(df[col].unique())
    print(df[col].count())

gender
<StringArray>
['male', 'unknown', 'female', 'non-binary']
Length: 4, dtype: str
10300
region
<StringArray>
['east', 'west', 'south', 'north']
Length: 4, dtype: str
10300
subscription_plan
<StringArray>
['basic', 'pro', 'plus']
Length: 3, dtype: str
10300
payment_method
<StringArray>
['paypal', 'e-wallet', 'credit_card', 'crypto', nan]
Length: 5, dtype: str
9575


## Handling Duplicates

In [84]:
print(df.duplicated().sum())
df1 = df.drop_duplicates().copy()
df1.duplicated().sum()

300


np.int64(0)

In [85]:
# Save Dataset
os.makedirs(data_dir,exist_ok=True)
# Set title
def save_csv(data,filename:str):
    df.to_csv(data_dir+filename+'.csv')
save_csv(df1,'customer_churn_clean')